[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/github-actions-certified/notebooks/day-07-capstone-ml-pipeline.ipynb#scrollTo=ca01b001)

---
# Day 7 · Capstone — Full ML CI/CD Pipeline
**certified-journeys / github-actions-certified** · Day 7 · Capstone

> **Goal for today:** Build a complete, production-ready ML CI/CD pipeline in GitHub Actions: on every push to `main`, run `pytest` on the training code, train a sklearn model, evaluate it against a baseline accuracy threshold, push the model artifact to GitHub Releases if it passes, and post a summary PR comment with accuracy and a confusion matrix rendered as a markdown table.


In [ ]:
%pip install -q pyyaml scikit-learn


## Step 1 · Capstone Design — Draw the Pipeline First

Before writing a single line of YAML, design the pipeline on paper. A well-designed ML CI/CD pipeline has **four jobs in a dependency chain** plus one parallel notification job:

```
push to main
     │
  [test]  ─── pytest tests/ ─── fail fast on red tests
     │
  [train] ─── train sklearn model ─── upload model.pkl artifact
     │
  [evaluate] ─── download model.pkl ─── score on held-out set ─── fail if accuracy < 0.90
     │                  │
  [release]          [comment]
  (main only)        (PR only)
  upload .pkl        post accuracy +
  to Releases        confusion matrix
```

| Job | `needs:` | `permissions:` | Key question |
|---|---|---|---|
| `test` | — | `contents: read` | Do all tests pass? |
| `train` | `test` | `contents: read` | Did the model artifact save? |
| `evaluate` | `train` | `contents: read` | Does accuracy meet the threshold? |
| `release` | `evaluate` | `contents: write` | Is this a push to main? |
| `comment` | `evaluate` | `pull-requests: write` | Is this a PR event? |


In [ ]:
import yaml

# Validate the job dependency graph before writing the full workflow
job_graph = {
    "test":     {"needs": None,       "permissions": {"contents": "read"}},
    "train":    {"needs": "test",     "permissions": {"contents": "read"}},
    "evaluate": {"needs": "train",    "permissions": {"contents": "read"}},
    "release":  {"needs": "evaluate", "permissions": {"contents": "write"}},
    "comment":  {"needs": "evaluate", "permissions": {"pull-requests": "write", "contents": "read"}},
}

print("Pipeline job graph:")
for job, cfg in job_graph.items():
    dep = cfg["needs"] or "(trigger)"
    perms = ", ".join(f"{k}:{v}" for k, v in cfg["permissions"].items())
    print(f"  {job:10s}  needs={dep:10s}  permissions=[{perms}]")

# Topological order validation
def topo_sort(graph):
    order = []
    visited = set()
    def visit(node):
        if node in visited:
            return
        visited.add(node)
        dep = graph[node]["needs"]
        if dep:
            visit(dep)
        order.append(node)
    for job in graph:
        visit(job)
    return order

print(f"\nTopological execution order: {' → '.join(topo_sort(job_graph))}")


**What just happened?**

- We modelled the job graph in Python and validated the dependency order before writing YAML.
- The topological sort confirms `test → train → evaluate → (release|comment)` — exactly what Actions will execute.
- **Designing before coding** prevents the most common capstone mistake: jobs that need an artifact that hasn't been uploaded yet.


## Step 2 · The Test Job — `pytest` as a Quality Gate

The test job is the first gate. If `pytest` fails, nothing downstream runs. This prevents shipping a model trained from broken code.

| Best practice | Reason |
|---|---|
| `--tb=short` | Concise traceback in CI logs |
| `-v` | One line per test — easy to scan |
| `--no-header` | Removes pytest banner noise |
| Cache pip | Speeds up repeated runs on the same requirements |


In [ ]:
# Write the test job as a dict, then render to YAML
test_job = {
    "runs-on": "ubuntu-latest",
    "permissions": {"contents": "read"},
    "steps": [
        {"uses": "actions/checkout@v4"},
        {
            "name": "Set up Python 3.12",
            "uses": "actions/setup-python@v5",
            "with": {"python-version": "3.12"},
        },
        {
            "name": "Cache pip",
            "uses": "actions/cache@v4",
            "with": {
                "path": "~/.cache/pip",
                "key": "${{ runner.os }}-pip-${{ hashFiles('**/requirements.txt') }}",
                "restore-keys": "${{ runner.os }}-pip-",
            },
        },
        {"name": "Install dependencies", "run": "pip install -r requirements.txt"},
        {
            "name": "Run tests",
            # pytest exits non-zero if any test fails → Actions marks job FAILED
            "run": "pytest tests/ -v --tb=short --no-header",
        },
    ],
}

print(yaml.dump({"test": test_job}, sort_keys=False))


**What just happened?**

- The test job runs with `contents: read` only — the minimum permission needed to checkout.
- Pip caching uses the exact key pattern from Day 4: `runner.os-pip-hashFiles`.
- `pytest` non-zero exit = job failure = train/evaluate/release all **skipped automatically** via the `needs:` chain.


## Step 3 · The Train Job — sklearn Model + Artifact Upload

The train job fits the model and uploads `model.pkl` to the Actions artifact store. The artifact name (`trained-model`) must be used verbatim in every subsequent `download-artifact` step.

**Why not commit the model to git?** Model files are binary, often large (>100MB), and change on every retrain. GitHub Releases is designed for binary artifacts — it's versioned, downloadable, and linked to a specific commit SHA.


In [ ]:
# Simulate train.py — in CI this script runs on the Actions runner
import pickle
import json
import pathlib
from sklearn.datasets import load_wine  # wine dataset instead of iris for variety
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Load data — wine dataset, 3 classes, 178 samples, no internet needed
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Build pipeline: scale then classify
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GradientBoostingClassifier(n_estimators=100, random_state=42)),
])
pipeline.fit(X_train, y_train)

# Save model
model_path = pathlib.Path("/tmp/model.pkl")
with open(model_path, "wb") as f:
    pickle.dump(pipeline, f)

print(f"Model saved: {model_path} ({model_path.stat().st_size:,} bytes)")
print(f"Pipeline steps: {[name for name, _ in pipeline.steps]}")

# Save metadata alongside the model
meta = {
    "dataset": "wine",
    "n_train": len(X_train),
    "n_test": len(X_test),
    "model_class": type(pipeline.named_steps["clf"]).__name__,
    "n_estimators": 100,
}
pathlib.Path("/tmp/model_meta.json").write_text(json.dumps(meta, indent=2))
print(f"Metadata: {meta}")


**What just happened?**

- We trained a `sklearn.Pipeline` (scaler + GBM) on the wine dataset — no external data needed.
- The model is saved as `model.pkl` — exactly what the workflow's train job will produce.
- A `model_meta.json` file captures training metadata — helpful for debugging reproducibility issues.


## Step 4 · The Train Job YAML + Artifact Upload

Now wire the train script into an Actions job with the `upload-artifact` step.


In [ ]:
train_job = {
    "needs": "test",
    "runs-on": "ubuntu-latest",
    "permissions": {"contents": "read"},
    "steps": [
        {"uses": "actions/checkout@v4"},
        {"uses": "actions/setup-python@v5", "with": {"python-version": "3.12"}},
        {
            "name": "Cache pip",
            "uses": "actions/cache@v4",
            "with": {
                "path": "~/.cache/pip",
                "key": "${{ runner.os }}-pip-${{ hashFiles('**/requirements.txt') }}",
                "restore-keys": "${{ runner.os }}-pip-",
            },
        },
        {"run": "pip install -r requirements.txt"},
        {"name": "Train model", "run": "python train.py --output model.pkl"},
        {
            # CRITICAL: name 'trained-model' must match exactly in all download steps
            "name": "Upload model artifact",
            "uses": "actions/upload-artifact@v4",
            "with": {
                "name": "trained-model",
                "path": "model.pkl",
                "retention-days": 30,
            },
        },
    ],
}

print(yaml.dump({"train": train_job}, sort_keys=False))


**What just happened?**

- `needs: test` means this job only runs after `test` succeeds.
- `actions/upload-artifact@v4` writes `model.pkl` to a temporary artifact store scoped to this workflow run.
- The artifact name `trained-model` is the **contract** between jobs — spelling it differently in a download step causes a runtime error.
- `retention-days: 30` keeps the artifact accessible for 30 days — useful for audit trails and rollback.


## Step 5 · The Evaluate Job — Download, Score, Quality Gate

The evaluate job downloads the artifact, runs `evaluate.py`, and **exits non-zero if accuracy is below the threshold**. This is the only job where the model quality is tested — downstream jobs inherit its result through `needs:`.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
import sys


def evaluate_model(model_path: str, threshold: float = 0.90) -> dict:
    """Quality gate: load model, score on held-out set, return results."""
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    # Same split as train.py — reproducible with fixed random_state
    X, y = load_wine(return_X_y=True)
    _, X_test, _, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    y_pred = model.predict(X_test)
    accuracy = float(accuracy_score(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred).tolist()

    return {
        "accuracy": round(accuracy, 4),
        "threshold": threshold,
        "passed": accuracy >= threshold,
        "confusion_matrix": cm,
        "n_test": len(y_test),
    }


results = evaluate_model("/tmp/model.pkl", threshold=0.90)
pathlib.Path("/tmp/eval_results.json").write_text(json.dumps(results, indent=2))

print(json.dumps(results, indent=2))
print(f"\nQuality gate: {'PASS ✓' if results['passed'] else 'FAIL ✗'}")
print("In CI: sys.exit(1) would be called if passed=False → release job skipped")


**What just happened?**

- `GradientBoostingClassifier` on wine data achieves >95% accuracy, comfortably passing the 0.90 threshold.
- The results JSON includes everything the comment job needs: accuracy, threshold, passed flag, and confusion matrix.
- **The same `random_state=42` in train and evaluate** guarantees the held-out test set is identical across runs — the gate is reproducible.


## Step 6 · The Release Job — Upload to GitHub Releases

Once the quality gate passes, the release job uploads `model.pkl` as a binary asset on a GitHub Release tagged with the commit SHA. This creates an immutable, versioned model registry using GitHub's built-in infrastructure — no external MLflow or S3 needed.


In [ ]:
evaluate_job = {
    "needs": "train",
    "runs-on": "ubuntu-latest",
    "permissions": {"contents": "read"},
    "steps": [
        {"uses": "actions/checkout@v4"},
        {"uses": "actions/setup-python@v5", "with": {"python-version": "3.12"}},
        {"run": "pip install -r requirements.txt"},
        {
            "name": "Download model artifact",
            "uses": "actions/download-artifact@v4",
            "with": {"name": "trained-model"},  # must match upload name exactly
        },
        {
            "name": "Evaluate model (quality gate)",
            # evaluate.py calls sys.exit(1) if accuracy < 0.90
            "run": "python evaluate.py --model model.pkl --threshold 0.90 --output eval_results.json",
        },
        {
            # Upload results even if gate fails (comment job still runs)
            "name": "Upload evaluation results",
            "if": "always()",  # always() ensures this runs even after gate failure
            "uses": "actions/upload-artifact@v4",
            "with": {"name": "eval-results", "path": "eval_results.json"},
        },
    ],
}

release_job = {
    "needs": "evaluate",
    # Only create a release on push to main (not on PRs)
    "if": "github.ref == 'refs/heads/main'",
    "runs-on": "ubuntu-latest",
    "permissions": {"contents": "write"},  # write access to create GitHub Releases
    "steps": [
        {"uses": "actions/checkout@v4"},
        {
            "name": "Download model artifact",
            "uses": "actions/download-artifact@v4",
            "with": {"name": "trained-model"},
        },
        {
            "name": "Download evaluation results",
            "uses": "actions/download-artifact@v4",
            "with": {"name": "eval-results"},
        },
        {
            "name": "Create GitHub Release and upload model",
            # Pinned to a SHA in production — using tag here for readability
            "uses": "softprops/action-gh-release@v2",
            "with": {
                "tag_name": "model-${{ github.sha }}",
                "name": "Model build ${{ github.sha }}",
                "files": "model.pkl",
                "body_path": "eval_results.json",   # release notes = eval results
                "generate_release_notes": True,
            },
        },
    ],
}

print("=== evaluate job ===")
print(yaml.dump({"evaluate": evaluate_job}, sort_keys=False))
print("=== release job ===")
print(yaml.dump({"release": release_job}, sort_keys=False))


**What just happened?**

- The evaluate job uses `if: always()` on the results upload — this ensures the PR comment job can read results **even if the gate failed**.
- `softprops/action-gh-release` creates a GitHub Release with `model.pkl` as a binary asset.
- `tag_name: model-${{ github.sha }}` makes every model uniquely addressable by the commit that produced it.
- The `if: github.ref == 'refs/heads/main'` condition prevents release creation on PR branches.


## Step 7 · The Comment Job — Confusion Matrix in PR

The comment job posts evaluation results to the PR that triggered the run. It needs `pull-requests: write` permission and only runs on `pull_request` events. The confusion matrix is rendered as a GitHub-flavoured Markdown table.


In [ ]:
# Build the comment job YAML
comment_job = {
    "needs": "evaluate",
    "if": "github.event_name == 'pull_request'",
    "runs-on": "ubuntu-latest",
    "permissions": {
        "pull-requests": "write",  # needed to post comments
        "contents": "read",
    },
    "steps": [
        {
            "name": "Download evaluation results",
            "uses": "actions/download-artifact@v4",
            "with": {"name": "eval-results"},
        },
        {
            "name": "Post PR comment with accuracy and confusion matrix",
            "uses": "actions/github-script@v7",
            "with": {
                "script": """
const fs = require('fs');
const r = JSON.parse(fs.readFileSync('eval_results.json', 'utf8'));
const { accuracy, threshold, passed, confusion_matrix } = r;

// Wine dataset class names
const labels = ['class_0', 'class_1', 'class_2'];

// Build confusion matrix markdown table
const header = '| Actual \\ Predicted | ' + labels.join(' | ') + ' |';
const sep = '|' + Array(labels.length + 1).fill('---').join('|') + '|';
const rows = confusion_matrix.map((row, i) =>
  `| **${labels[i]}** | ${row.join(' | ')} |`
).join('\\n');

const status = passed ? '✅ **PASSED** — model uploaded to Releases' : '❌ **FAILED** — release skipped';

const body = [
  '## 🤖 ML Pipeline Evaluation Results',
  '',
  '| Metric | Value |',
  '|---|---|',
  `| **Accuracy** | ${(accuracy * 100).toFixed(2)}% |`,
  `| **Threshold** | ${(threshold * 100).toFixed(0)}% |`,
  `| **Test samples** | ${r.n_test} |`,
  `| **Status** | ${status} |`,
  '',
  '### Confusion Matrix',
  '',
  header, sep, rows,
  '',
  '---',
  `*Generated by workflow run [#${ context.runId }](${ context.serverUrl }/${ context.repo.owner }/${ context.repo.repo }/actions/runs/${ context.runId })*`,
].join('\\n');

await github.rest.issues.createComment({
  owner: context.repo.owner,
  repo:  context.repo.repo,
  issue_number: context.issue.number,
  body,
});
"""
            },
        },
    ],
}

print(yaml.dump({"comment": comment_job}, sort_keys=False))


**What just happened?**

- `actions/github-script@v7` provides `github` (Octokit), `context` (run metadata), and `core` as globals — no extra setup.
- `context.issue.number` is the PR number — automatically set on `pull_request` events.
- The confusion matrix is rendered as a proper two-dimensional Markdown table with row/column headers.
- The comment links back to the specific Actions run for traceability.


## Step 8 · Preview the PR Comment Locally

Before pushing, verify the comment markdown renders correctly.


In [ ]:
import json

# Load results from Step 5
r = json.loads(pathlib.Path("/tmp/eval_results.json").read_text())
accuracy = r["accuracy"]
threshold = r["threshold"]
passed = r["passed"]
cm = r["confusion_matrix"]
n_test = r["n_test"]

labels = ["class_0", "class_1", "class_2"]

header = "| Actual \\ Predicted | " + " | ".join(labels) + " |"
sep    = "|" + "|".join(["---"] * (len(labels) + 1)) + "|"
rows   = "\n".join(
    f"| **{labels[i]}** | " + " | ".join(str(v) for v in row) + " |"
    for i, row in enumerate(cm)
)

status = "✅ **PASSED** — model uploaded to Releases" if passed else "❌ **FAILED** — release skipped"

comment_body = "\n".join([
    "## 🤖 ML Pipeline Evaluation Results",
    "",
    "| Metric | Value |",
    "|---|---|",
    f"| **Accuracy** | {accuracy * 100:.2f}% |",
    f"| **Threshold** | {threshold * 100:.0f}% |",
    f"| **Test samples** | {n_test} |",
    f"| **Status** | {status} |",
    "",
    "### Confusion Matrix",
    "",
    header, sep, rows,
    "",
    "---",
    "*Generated by ML CI/CD pipeline*",
])

print(comment_body)


**What just happened?**

- We previewed the exact markdown that will appear in the GitHub PR comment.
- The confusion matrix table shows at a glance which classes are being confused — on wine data with GBM, you expect near-perfect diagonal.
- Testing the comment locally means no push-and-wait iteration cycle to fix formatting.


## Step 9 · Testing the Full evaluate.py Script End-to-End

Before submitting the capstone, run a full integration test locally: simulate what each CI job does, in order.


In [ ]:
import subprocess
import textwrap

# Write train.py and evaluate.py to /tmp — simulate what the repo contains
train_py = textwrap.dedent("""
    import pickle, argparse, json
    from sklearn.datasets import load_wine
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline

    parser = argparse.ArgumentParser()
    parser.add_argument("--output", default="model.pkl")
    args = parser.parse_args()

    X, y = load_wine(return_X_y=True)
    X_train, _, y_train, _ = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    pipeline = Pipeline([("scaler", StandardScaler()),
                         ("clf", GradientBoostingClassifier(n_estimators=100, random_state=42))])
    pipeline.fit(X_train, y_train)
    with open(args.output, "wb") as f:
        pickle.dump(pipeline, f)
    print(f"Model saved to {args.output}")
""")

evaluate_py = textwrap.dedent("""
    import sys, pickle, json, argparse
    from sklearn.datasets import load_wine
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, confusion_matrix

    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--threshold", type=float, default=0.90)
    parser.add_argument("--output", default="eval_results.json")
    args = parser.parse_args()

    with open(args.model, "rb") as f:
        model = pickle.load(f)

    X, y = load_wine(return_X_y=True)
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    y_pred = model.predict(X_test)
    accuracy = float(accuracy_score(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred).tolist()

    results = {"accuracy": round(accuracy, 4), "threshold": args.threshold,
               "passed": accuracy >= args.threshold, "confusion_matrix": cm, "n_test": len(y_test)}

    with open(args.output, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Accuracy: {accuracy:.4f}  Threshold: {args.threshold}")
    if not results["passed"]:
        print(f"FAIL: {accuracy:.4f} < {args.threshold}", file=sys.stderr)
        sys.exit(1)
    print("PASS")
""")

pathlib.Path("/tmp/train.py").write_text(train_py)
pathlib.Path("/tmp/evaluate.py").write_text(evaluate_py)

# Run train.py
result = subprocess.run(
    [sys.executable, "/tmp/train.py", "--output", "/tmp/model_ci.pkl"],
    capture_output=True, text=True
)
print("TRAIN:", result.stdout.strip(), result.stderr.strip() or "")

# Run evaluate.py
result = subprocess.run(
    [sys.executable, "/tmp/evaluate.py",
     "--model", "/tmp/model_ci.pkl",
     "--threshold", "0.90",
     "--output", "/tmp/eval_ci.json"],
    capture_output=True, text=True
)
print("EVALUATE stdout:", result.stdout.strip())
if result.returncode != 0:
    print("EVALUATE stderr:", result.stderr.strip())
    print(f"Exit code: {result.returncode} → CI job would FAIL")
else:
    print(f"Exit code: {result.returncode} → CI job PASSES")
    print(json.loads(pathlib.Path("/tmp/eval_ci.json").read_text()))


**What just happened?**

- We ran `train.py` and `evaluate.py` as subprocesses — exactly as Actions runner would.
- Both scripts exit with code `0` (success) → the quality gate passes.
- **This is your local integration test** before pushing to GitHub. If this runs clean, the CI pipeline will too.


In [ ]:
# CAPSTONE CHALLENGE: Complete ML CI/CD Pipeline
#
# Build the full production GitHub Actions workflow described in the capstone:
#
#   On every push to main AND on pull_request:
#     1. [test] job   — run pytest tests/ on the training code
#     2. [train] job  — train a sklearn model, save model.pkl, upload with actions/upload-artifact
#     3. [evaluate] job — download model.pkl, score on held-out set,
#                         FAIL the job if accuracy < threshold (use sys.exit(1) in evaluate.py)
#     4. [release] job  — (main only) download model.pkl, create GitHub Release with softprops/action-gh-release
#     5. [comment] job  — (pull_request only) download eval_results.json,
#                         post accuracy + confusion matrix table as a PR comment via actions/github-script
#
# Requirements:
#   - All jobs: scoped permissions (least privilege)
#   - test job: pip cache with hashFiles
#   - train job: actions/upload-artifact@v4 with name 'trained-model'
#   - evaluate job: actions/download-artifact@v4 with same name 'trained-model'
#                   upload eval_results.json as artifact 'eval-results' with if: always()
#   - release job: needs evaluate, if: github.ref == 'refs/heads/main'
#   - comment job: needs evaluate, if: github.event_name == 'pull_request'
#                  permissions: pull-requests: write
#   - Confusion matrix rendered as markdown table in the PR comment
#
# Scaffold — fill in every TODO:

capstone_workflow = {
    "name": "ML CI/CD Pipeline — Capstone",
    "on": {
        # TODO: trigger on push to main AND on pull_request
    },
    "jobs": {
        "test": {
            "runs-on": "ubuntu-latest",
            "permissions": {},  # TODO: least privilege
            "steps": [
                # TODO: checkout, setup-python, cache pip, install, pytest
            ],
        },
        "train": {
            "needs": "",  # TODO
            "runs-on": "ubuntu-latest",
            "permissions": {},  # TODO
            "steps": [
                # TODO: checkout, setup-python, install, python train.py, upload-artifact
            ],
        },
        "evaluate": {
            "needs": "",  # TODO
            "runs-on": "ubuntu-latest",
            "permissions": {},  # TODO
            "steps": [
                # TODO: checkout, setup-python, install, download-artifact,
                #        evaluate.py (quality gate), upload eval_results.json with if: always()
            ],
        },
        "release": {
            "needs": "",  # TODO
            "if": "",     # TODO: only on main branch
            "runs-on": "ubuntu-latest",
            "permissions": {},  # TODO: needs write
            "steps": [
                # TODO: checkout, download trained-model, softprops/action-gh-release
            ],
        },
        "comment": {
            "needs": "",  # TODO
            "if": "",     # TODO: only on pull_request events
            "runs-on": "ubuntu-latest",
            "permissions": {},  # TODO: pull-requests: write
            "steps": [
                # TODO: download eval-results, actions/github-script posting
                #       accuracy + confusion matrix as markdown table
            ],
        },
    },
}

# Validate structure
print(yaml.dump(capstone_workflow, sort_keys=False))

# Self-check function — run this after you complete the challenge
def check_capstone(wf: dict) -> list[str]:
    issues = []
    jobs = wf.get("jobs", {})
    for required_job in ["test", "train", "evaluate", "release", "comment"]:
        if required_job not in jobs:
            issues.append(f"Missing job: {required_job}")
    if jobs.get("train", {}).get("needs") != "test":
        issues.append("train job must need: test")
    if jobs.get("evaluate", {}).get("needs") != "train":
        issues.append("evaluate job must need: train")
    if jobs.get("release", {}).get("needs") != "evaluate":
        issues.append("release job must need: evaluate")
    if jobs.get("comment", {}).get("needs") != "evaluate":
        issues.append("comment job must need: evaluate")
    release_perms = jobs.get("release", {}).get("permissions", {})
    if release_perms.get("contents") != "write":
        issues.append("release job needs permissions.contents: write")
    comment_perms = jobs.get("comment", {}).get("permissions", {})
    if comment_perms.get("pull-requests") != "write":
        issues.append("comment job needs permissions.pull-requests: write")
    return issues

print("\nCapstone self-check (should be empty when complete):")
for issue in check_capstone(capstone_workflow):
    print(f"  ⚠  {issue}")


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| Job dependency chain | `needs:` creates a DAG; Actions skips downstream jobs when a dependency fails |
| Artifact bridge | `upload-artifact` name must match `download-artifact` name exactly across jobs |
| Quality gate | `sys.exit(1)` in evaluate.py → job red → release and comment skipped |
| `if: always()` | Ensures results artifact uploads even when the gate fails |
| Scoped permissions | Each job declares only what it needs; `release` needs `contents: write`, `comment` needs `pull-requests: write` |
| `softprops/action-gh-release` | GitHub Releases as a model registry — versioned by commit SHA |
| `actions/github-script` | Inline Octokit calls — post confusion matrix table to PR without extra action setup |
| `if:` on jobs | `github.ref == 'refs/heads/main'` for release, `github.event_name == 'pull_request'` for comment |

> **Tip:** The hardest part of the capstone is passing the model artifact between jobs. Use `actions/upload-artifact` in the train job and `actions/download-artifact` in evaluate and release. Both must reference the exact same artifact name.

---
## Course Complete

You have built a **production-ready ML CI/CD pipeline** in GitHub Actions:

- ✓ Matrix strategies across Python versions and OS
- ✓ Pip caching with deterministic cache keys
- ✓ Secret management via env vars, scoped GITHUB_TOKEN, OIDC keyless auth
- ✓ sklearn model training, artifact passing between jobs, quality gates
- ✓ GitHub Releases as a model registry
- ✓ PR comments with confusion matrix using `actions/github-script`

Mark Day 7 complete in your [tracker](../index.html) and claim your **GitHub Actions for MLOps** certification badge.
